# Military Strikes Forecasting

Generate a forecasting dataset about global military strikes and attack operations using the LightningRod SDK. Fine-tune a model via RL that outperforms frontier LLMs on strike prediction.

In [9]:
%pip install lightningrod-ai python-dotenv pandas

from IPython.display import clear_output
clear_output()

from datetime import datetime

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

True

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/sign-up?redirect=/api) to get your API key and **$50 of free credits**.

In [10]:
from lightningrod import LightningRod
from lightningrod.utils import config

api_key = config.get_config_value("LIGHTNINGROD_API_KEY")
lr = LightningRod(api_key=api_key)

## Build the pipeline

Configure the pipeline with domain-specific instructions and examples for military strike forecasting. Covers airstrikes, missile strikes, drone strikes, and naval strikes across state and non-state actors globally.

In [11]:
instructions = """
Generate binary forecasting questions specifically about military strikes and attack operations.

Cover all strike types:
- Airstrikes (fighter jets, bombers, helicopter gunships)
- Missile strikes (ballistic, cruise, hypersonic)
- Drone strikes (kamikaze drones, armed UAVs, drone swarms)
- Naval strikes (ship-launched missiles, naval gunfire, submarine attacks)

Cover both state and non-state actors. Use the natural language of news reporting:
- State actors: country names, leader names, named military units (IDF, IRGC, Pentagon)
- Non-state actors: group names (Hamas, Houthis, Hezbollah, ISIS, Wagner)

Questions must:
- Name the specific actor conducting the strike
- Name the specific target (location, infrastructure, military asset, or group)
- Have a specific date or event milestone as resolution criteria
- Be objectively verifiable from open-source news
- Span the full probability spectrum \u2014 some likely, some unlikely, some that won't happen
"""

good_examples = [
    "Will the IDF conduct airstrikes on Hezbollah weapons depots in the Bekaa Valley before November 2024?",
    "Will US Air Force B-52s conduct strikes on Houthi military infrastructure in Yemen before March 2024?",
    "Will Russian Su-34s carry out airstrikes on Kharkiv civilian infrastructure before June 2024?",
    "Will Iran launch a direct ballistic missile strike on Israeli territory before May 2024?",
    "Will Houthi forces fire anti-ship missiles at US Navy destroyers in the Red Sea before February 2024?",
    "Will Ukraine conduct drone strikes on Russian oil refineries inside Russian territory before April 2024?",
    "Will North Korea launch an ICBM test over Japanese waters before January 2024?",
    "Will Houthi forces attack a commercial vessel with drone boats in the Red Sea before March 2024?",
    "Will the US Navy conduct ship-launched Tomahawk strikes on Houthi radar sites before February 2024?",
    "Will Russian forces launch an armored offensive toward Chasiv Yar before April 2024?",
    "Will Israel conduct airstrikes on Iranian nuclear facilities at Natanz before December 2024?",
    "Will NATO aircraft conduct strikes inside Russian territory before January 2025?",
    "Will China conduct missile strikes on Taiwanese military bases before the end of 2024?",
    "Will Pakistan conduct airstrikes on Afghan Taliban positions before March 2024?",
]

bad_examples = [
    "Will there be an attack somewhere? (no specific actor, target, or location)",
    "Will violence increase in the Middle East? (vague, not a specific strike event)",
    "Will conflict continue in Ukraine? (trivially obvious, not a specific strike)",
    "Will missiles be fired? (no actor, no target, no date)",
    "Will tensions escalate? (not a verifiable strike event)",
    "Will there be drone activity near the border? (too vague to verify)",
    "Will the situation get worse? (subjective, not measurable)",
    "Will airstrikes happen in 2025? (no actor, no target, too broad)",
    "Will someone retaliate? (no specific actor or method)",
    "Will the war end? (not a strike event, different question type)",
]

search_queries = [
    "military airstrike",
    "military strike",
    "missile strike",
    "drone strike",
    "naval strike",
]

In [12]:
from lightningrod import (
    BinaryAnswerType,
    NewsSeedGenerator,
    ForwardLookingQuestionGenerator,
    NewsContextGenerator,
    WebSearchLabeler,
    QuestionPipeline,
)

answer_type = BinaryAnswerType()

pipeline = QuestionPipeline(
    seed_generator=NewsSeedGenerator(
        start_date=datetime(2024, 6, 1),
        end_date=datetime(2026, 3, 1),
        interval_duration_days=7,
        search_query=search_queries,
        articles_per_search=10,
    ),
    question_generator=ForwardLookingQuestionGenerator(
        instructions=instructions,
        examples=good_examples,
        bad_examples=bad_examples,
        answer_type=answer_type,
        questions_per_seed=5,
    ),
    context_generators=[
        NewsContextGenerator(
            articles_per_query=3,
            num_search_queries=3,
            num_articles=5,
        )
    ],
    labeler=WebSearchLabeler(answer_type=answer_type),
)

## Run the pipeline

Collect news articles, generate questions, and label answers. Set `max_questions=10000` for a full production run; reduce for testing.

In [13]:
dataset = lr.transforms.run(pipeline, max_questions=500, name="Military strikes forecasting")

samples = dataset.download()
pct = (sum(1 for s in samples if s.is_valid is True) / len(samples) * 100) if samples else 0
print(f"{len(samples)} samples ({pct:.1f}% valid)")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Pipeline Completed                                                                                          │
│                                                                                                                 │
│    Job ID:           cd0c7523-c8a2-4189-9170-5ff50d19b787                                                       │
│                                                                                                                 │
│    Total cost: $3.90                                                                                            │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━┳━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓  │
│  ┃ Step               ┃ Progress             ┃  In ┃ Out ┃ Rejected ┃ Errors ┃ Rejection Reasons  ┃ Duration ┃  │
│  ┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━╇━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩  │
│  │ NewsSeedGenerator… │ Complete             │  10 │  85 │        0 │      0 │ -                  │       1s │  │
│  │ ForwardLookingQue… │ Complete             │  85 │ 396 │       29 │      0 │ date_close not     │       3s │  │
│  │                    │                      │     │     │          │        │ after event_date   │          │  │
│  │                    │                      │     │     │          │        │ (29)               │          │  │
│  │ KeyDeduplicationT… │ Complete             │ 396 │ 391 │        5 │      0 │ Duplicate matched: │       0s │  │
│  │                    │                      │     │     │          │        │ {'date_close':     │          │  │
│  │                    │                      │     │     │          │        │ '2024-10-01        │          │  │
│  │                    │                      │     │     │          │        │ 00:00:00',         │          │  │
│  │                    │                      │     │     │          │        │ 'question_text':   │          │  │
│  │                    │                      │     │     │          │        │ "'will hezbollah   │          │  │
│  │                    │                      │     │     │          │        │ launch a rocket or │          │  │
│  │                    │                      │     │     │          │        │ missile attack     │          │  │
│  │                    │                      │     │     │          │        │ that reaches the   │          │  │
│  │                    │                      │     │     │          │        │ city of tel aviv,  │          │  │
│  │                    │                      │     │     │          │        │ israel, before     │          │  │
│  │                    │                      │     │     │          │        │ october 1, 2024?'  │          │  │
│  │                    │                      │     │     │          │        │ (ratio=1.00)"}     │          │  │
│  │                    │                      │     │     │          │        │ (1), Duplicate     │          │  │
│  │                    │                      │     │     │          │        │ matched:           │          │  │
│  │                    │                      │     │     │          │        │ {'date_close':     │          │  │
│  │                    │                      │     │     │          │        │ '2024-11-01        │          │  │
│  │                    │                      │     │     │          │        │ 00:00:00',         │          │  │
│  │                    │                      │     │     │          │        │ 'question_text':   │          │  │
│  │                    │                      │     │  

425 samples (85.9% valid)


## Prepare the dataset

Filter valid samples, deduplicate, and split into train/test sets using a temporal strategy.

In [14]:
from lightningrod import prepare_for_training, FilterParams, SplitParams

train_dataset, test_dataset = prepare_for_training(
    dataset,
    filter=FilterParams(days_to_resolution_range=(1, 90)),
    split=SplitParams(test_size=0.2),
)

for name, ds in [("Train", train_dataset), ("Test", test_dataset)]:
    data = ds.flattened()
    yes_count = sum(1 for s in data if s.get("label") in (1, "1", 1.0))
    print(f"{name}: {len(data)} rows, {yes_count/len(data)*100:.1f}% yes")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> prepare_for_training                                                                                        │
│                                                                                                                 │
│    Starting with 425 samples                                                                                    │
│                                                                                                                 │
│    Filter:  Dropped 60 invalid, 157 horizon → 208 remain                                                        │
│    Dedup:   208 remain (0 duplicates)                                                                           │
│    Split:   Splits: 150 train | 42 test (0 dropped, no prediction_date)                                         │
│             16 train samples removed for leakage                                                                │
│                                                                                                                 │
│  ⚠ Unhealthy dataset                                                                                            │
│                                                                                                                 │
│  Only 150 train samples remain after preparation. This is below the recommended minimum of 200 for effective    │
│  training.                                                                                                      │
│                                                                                                                 │
│    Tips:                                                                                                        │
│      • Increase max_questions in lr.transforms.run() to generate more samples.                                  │
│      • Increase questions_per_seed in your question generator (ForwardLookingQuestionGenerator or               │
│  QuestionGenerator) to produce more questions from each seed article.Add more search queries to your seed       │
│  generator to diversify seed sources.                                                                           │
│      • Widen the seed generator date range (start_date to end_date) to capture more events.                     │
│                                                                                                                 │
│  Only 42 test samples remain after preparation. This is below the recommended minimum of ~50 for reliable       │
│  evaluation.                                                                                                    │
│                                                                                                                 │
│    Tips:                                                                                                        │
│      • Generate more samples overall — test samples come from the most recent portion of your date range.       │
│      • Ensure your seed generator date range extends close to the present so recent events appear in the test   │
│  set.                                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Train: 150 rows, 46.7% yes
Test: 42 rows, 42.9% yes


## Train the model

Fine-tune `openai/gpt-oss-120b` via RL using the training parameters from our golf and WWTD experiments.

In [15]:
from lightningrod import GRPOTrainingConfig
BATCH_SIZE = 32                                                                                                                           

train_data = train_dataset.flattened()                                                                                                    
training_steps = max(10, len(train_data) // BATCH_SIZE)

training_config = GRPOTrainingConfig(
    base_model_id="openai/gpt-oss-120b",
    training_steps=training_steps,
    lora_rank=32,
    batch_size=BATCH_SIZE,
    num_rollouts=8,
    max_response_length=16384,
    learning_rate=4e-5,
)

cost_estimate = lr.training.estimate_cost(training_config, dataset=train_dataset)
print(f"Estimated cost: ${cost_estimate.total_cost_dollars:.2f}")
print(f"Effective steps: {cost_estimate.effective_steps}")
print(f"Train tokens: {cost_estimate.train_tokens:,}")
if cost_estimate.notes:
    print(f"Notes: {cost_estimate.notes}")

Estimated cost: $0.97
Effective steps: 5
Train tokens: 1,354,589
Notes: Estimate uses per-answer-type output token estimates; actual may vary


In [16]:
job = lr.training.run(training_config, dataset=train_dataset, name="military-strikes-v1")
print(f"Job {job.id} completed with status: {job.status}")
print(f"Model ID: {job.model_id}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Training COMPLETED                                                                                          │
│                                                                                                                 │
│    Job ID: 3a16bf58-6e3c-42af-9d83-34a9fddddfc3                                                                 │
│                                                                                                                 │
│    Model: checkpoint:3a16bf58-6e3c-42af-9d83-34a9fddddfc3                                                       │
│                                                                                                                 │
│    Model: checkpoint:3a16bf58-6e3c-42af-9d83-34a9fddddfc3                                                       │
│                                                                                                                 │
│    Reward: latest -0.6761  avg -0.6946  (5 steps)  (higher is better)                                           │
│                                                                                                                 │
│    Cost:  $1.42                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Job 3a16bf58-6e3c-42af-9d83-34a9fddddfc3 completed with status: COMPLETED
Model ID: checkpoint:3a16bf58-6e3c-42af-9d83-34a9fddddfc3


## Evaluate

Run the trained model against the test set, benchmarked against GPT-5.4.

In [20]:
from lightningrod import EvalModel, training

eval_job = lr.evals.run(
    training_config,
    job,
    test_dataset,
    extra_models=[
        EvalModel(model_id="openai/gpt-5.4", label="GPT-5.4"),
    ],
)

training.print_eval(eval_job)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Eval COMPLETED                                                                                              │
│                                                                                                                 │
│    Job ID: 4f7c056e-e655-4bc2-a12c-59ca4b6436b9                                                                 │
│    Dataset: a3a5efcc-9f0b-4498-bc3b-d98c62376352                                                                │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━┓                                                       │
│  ┃ Metric              ┃    Base ┃ GPT-5.4 ┃ Fine-tuned ┃                                                       │
│  ┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━┩                                                       │
│  │ brier_score         │  0.2131 │  0.2576 │     0.2051 │                                                       │
│  │ ece                 │  0.2237 │  0.1898 │     0.1576 │                                                       │
│  │ mean_reward         │ -0.9533 │ -0.7220 │    -0.5850 │                                                       │
│  │ mean_valid_reward   │ -0.6009 │ -0.7220 │    -0.5850 │                                                       │
│  │ n_samples           │      42 │      42 │         42 │                                                       │
│  │ n_valid             │      40 │      42 │         42 │                                                       │
│  │ parse_rate          │  0.9524 │  1.0000 │     1.0000 │                                                       │
│  │ total_cost          │       — │       — │     0.0166 │                                                       │
│  │ total_input_tokens  │   46484 │   43899 │      45411 │                                                       │
│  │ total_output_tokens │   13893 │     546 │      19111 │                                                       │
│  └─────────────────────┴─────────┴─────────┴────────────┘                                                       │
│                                                                                                                 │
│    Cost:  $0.02                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Eval COMPLETED                                                                                              │
│                                                                                                                 │
│    Job ID: 4f7c056e-e655-4bc2-a12c-59ca4b6436b9                                                                 │
│    Dataset: a3a5efcc-9f0b-4498-bc3b-d98c62376352                                                                │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━┓                                                       │
│  ┃ Metric              ┃    Base ┃ GPT-5.4 ┃ Fine-tuned ┃                                                       │
│  ┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━┩                                                       │
│  │ brier_score         │  0.2131 │  0.2576 │     0.2051 │                                                       │
│  │ ece                 │  0.2237 │  0.1898 │     0.1576 │                                                       │
│  │ mean_reward         │ -0.9533 │ -0.7220 │    -0.5850 │                                                       │
│  │ mean_valid_reward   │ -0.6009 │ -0.7220 │    -0.5850 │                                                       │
│  │ n_samples           │      42 │      42 │         42 │                                                       │
│  │ n_valid             │      40 │      42 │         42 │                                                       │
│  │ parse_rate          │  0.9524 │  1.0000 │     1.0000 │                                                       │
│  │ total_cost          │       — │       — │     0.0166 │                                                       │
│  │ total_input_tokens  │   46484 │   43899 │      45411 │                                                       │
│  │ total_output_tokens │   13893 │     546 │      19111 │                                                       │
│  └─────────────────────┴─────────┴─────────┴────────────┘                                                       │
│                                                                                                                 │
│    Cost:  $0.02                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
# Quick inference test
print(lr.predict(job.model_id, "Will Israel conduct airstrikes in southern Lebanon before April 2026?"))